# Reproducibility Notebook: Cluster-State Holographic Sequential Simulation

## Overview

This notebook reproduces the main results of the **cluster_state_holographic_sim** study, which verifies the 1D cluster-state MPS transfer unitary in cqed_sim, compares five decomposition strategies into native bosonic primitives, and benchmarks against direct GRAPE optimization.

**Problem Class:** DES | ANA | OPT

**Decomposition Strategies:**
- A: D+R+SNAP (F=0.500 — SNAP alone cannot entangle)
- B: D+SQR+CP (F=1.000 ideal, but collapses to F=0.0747 at N_cav=12)
- C: SQR+CP only (F=0.500 — no Fock mixing without Displacement)
- D: D+R+FreeEvolveCondPhase (best bounded: F=0.5465 at N_cav=12)
- E: GRAPE 400ns (F=0.999 — only physically validated high-fidelity route)

**Key conclusion:** GRAPE remains the only physically validated high-fidelity route; native decompositions fail at larger cavity dimensions.

## 1. Environment Setup

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

STUDY_ROOT = Path(".").resolve().parent
DATA_DIR = STUDY_ROOT / "data"
FIG_DIR = STUDY_ROOT / "figures"
ARTIFACT_DIR = STUDY_ROOT / "artifacts"

print(f"Study root: {STUDY_ROOT}")
print(f"Data files: {sorted(p.name for p in DATA_DIR.iterdir() if p.is_file())[:10]}")
print(f"Artifacts: {sorted(p.name for p in ARTIFACT_DIR.iterdir() if p.is_file())}")

## 2. Load Target Unitary

The target unitary is the cluster-state MPS transfer matrix, generated by `cqed_sim.unitary_synthesis.targets.make_target('cluster', n_match=1)`. It is saved as `artifacts/target_unitary.npz`.

In [ ]:
target_path = ARTIFACT_DIR / "target_unitary.npz"
if target_path.exists():
    target_data = np.load(target_path, allow_pickle=True)
    print("Target unitary arrays:", list(target_data.keys()))
    for k in target_data.keys():
        arr = target_data[k]
        print(f"  {k}: shape={arr.shape}, dtype={arr.dtype}")
        if arr.size <= 16:
            print(f"    {arr}")
else:
    print("Target unitary file not found")

## 3. Load Best Decomposition Artifacts

The artifacts directory contains the best decomposition results for Strategies B and D, plus the overall best decomposition.

In [ ]:
artifact_files = sorted(ARTIFACT_DIR.glob("*.json"))
for af in artifact_files:
    with open(af, "r") as f:
        data = json.load(f)
    print(f"\n--- {af.name} ---")
    if isinstance(data, dict):
        for k in list(data.keys())[:8]:
            val = data[k]
            if isinstance(val, (int, float, str, bool)):
                print(f"  {k}: {val}")
            else:
                print(f"  {k}: {type(val).__name__}")

## 4. Verify Wigner Comparison

The `wigner_comparison.npz` artifact contains cavity Wigner function data used to visually validate decomposition quality.

In [ ]:
wigner_path = ARTIFACT_DIR / "wigner_comparison.npz"
if wigner_path.exists():
    wigner_data = np.load(wigner_path, allow_pickle=True)
    print("Wigner comparison arrays:")
    for k in wigner_data.keys():
        arr = wigner_data[k]
        print(f"  {k}: shape={arr.shape}, dtype={arr.dtype}")
else:
    print("Wigner comparison file not found")

## 5. Load Iteration Results

The study evolved through 4 iterations, each refining the analysis. Key iteration data shows how the understanding progressed from ideal-mode results (iterations 1-2) to full-drift validation (iterations 3-4).

In [ ]:
iteration_files = sorted(DATA_DIR.glob("iteration*.json"))
print(f"Iteration result files ({len(iteration_files)}):")
for itf in iteration_files[:5]:
    with open(itf, "r") as f:
        data = json.load(f)
    print(f"  {itf.name}: {list(data.keys())[:5]}")

## 6. Reproduce Key Figures

In [ ]:
from IPython.display import display, Image

figure_files = sorted(FIG_DIR.glob("*.png"))
print(f"Available figures ({len(figure_files)}):")
for fig_path in figure_files[:10]:  # Show first 10
    print(f"\n--- {fig_path.stem} ---")
    display(Image(filename=str(fig_path), width=700))

if len(figure_files) > 10:
    print(f"\n... and {len(figure_files) - 10} more figures")

## 7. Summary

This notebook verified the key results of the cluster-state holographic simulation study:

1. **Target unitary** loaded from `artifacts/target_unitary.npz` — cluster MPS transfer matrix
2. **Best decompositions** (Strategies B, D) loaded from JSON artifacts
3. **Wigner comparison** data available for visual validation
4. **Iteration progression** traceable through `data/iteration*.json` files
5. **Publication figures** displayed (30+ figures)

**Strategy comparison at N_cav=12:**
| Strategy | Ideal F | Embedded F (N_cav=12) |
|----------|---------|----------------------|
| A: D+R+SNAP | 0.500 | — |
| B: D+SQR+CP | 1.000 | 0.0747 |
| D: D+R+FE | 0.9999 | 0.5465 |
| E: GRAPE 400ns | 0.999 | validated |

**Main conclusion:** Only GRAPE provides physically validated high-fidelity gates. Native decompositions suffer from truncation artifacts that collapse performance at realistic cavity dimensions.

### To fully re-run from scratch:
```python
# The study evolved through multiple scripts; the latest is:
# python run_study_v4.py         # Latest iteration
# python make_figures.py         # Generate figures
```